In [ ]:
# Retail Sales Prediction using Scikit-learn Pipeline

## Objective
Build a reproducible regression pipeline to predict `items_sold` at a retail store using historical transaction data.

## Dataset Columns
- transaction_date, store_id, store_size, location_type, promotion_type, is_weekend, is_festival, competition_density, items_sold

---

## Task 1: Date Feature Engineering 

```python
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load the dataset
df = pd.read_csv('q3_retail_promotions.csv')

print("="*60)
print("DATASET INFORMATION")
print("="*60)
print(f"\nDataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nFirst 5 rows:")
print(df.head())

print("\n" + "-"*40)
print("Data Types:")
print("-"*40)
print(df.dtypes)

# Convert transaction_date to datetime
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

# Extract date features
df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.dayofweek  # Monday=0, Sunday=6

# Create is_month_end feature (day of month >= 25)
df['is_month_end'] = (df['transaction_date'].dt.day >= 25).astype(int)

print("\n" + "="*60)
print("DATE FEATURE ENGINEERING RESULTS")
print("="*60)

print("\nNew features created:")
print("  - year: Year of transaction")
print("  - month: Month of transaction (1-12)")
print("  - day_of_week: Day of week (0=Monday, 6=Sunday)")
print("  - is_month_end: Binary flag for month end (day >= 25)")

print("\n" + "-"*40)
print("Sample of resulting dataframe (first 10 rows):")
print("-"*40)

# Select relevant columns for display
display_cols = ['transaction_date', 'year', 'month', 'day_of_week', 'is_month_end', 
                'store_id', 'store_size', 'promotion_type', 'items_sold']
print(df[display_cols].head(10))

print("\n" + "-"*40)
print("Date feature distributions:")
print("-"*40)
print("\nYear distribution:")
print(df['year'].value_counts().sort_index())
print("\nMonth distribution:")
print(df['month'].value_counts().sort_index())
print("\nDay of week distribution:")
print(df['day_of_week'].value_counts().sort_index())
print("\nis_month_end distribution:")
print(df['is_month_end'].value_counts())

# Verify no missing values after feature engineering
print("\n" + "-"*40)
print("Missing values after feature engineering:")
print("-"*40)
print(df.isnull().sum

## Task 2: Temporal Train-Test Split
print("="*60)
print("TEMPORAL TRAIN-TEST SPLIT")
print("="*60)

# Sort data by transaction_date
df_sorted = df.sort_values('transaction_date').reset_index(drop=True)

print(f"\nData sorted by transaction_date")
print(f"Date range: {df_sorted['transaction_date'].min()} to {df_sorted['transaction_date'].max()}")
print(f"Total records: {len(df_sorted)}")

# Use most recent 20% as test set
test_size = 0.2
split_idx = int(len(df_sorted) * (1 - test_size))

train_df = df_sorted.iloc[:split_idx].copy()
test_df = df_sorted.iloc[split_idx:].copy()

print(f"\nTraining set: {len(train_df)} records ({len(train_df)/len(df_sorted)*100:.1f}%)")
print(f"  Date range: {train_df['transaction_date'].min()} to {train_df['transaction_date'].max()}")
print(f"Test set: {len(test_df)} records ({len(test_df)/len(df_sorted)*100:.1f}%)")
print(f"  Date range: {test_df['transaction_date'].min()} to {test_df['transaction_date'].max()}")

# Verify no data leakage
print("\n" + "-"*40)
print("Verification - No overlap between train and test dates:")
print("-"*40)
train_max_date = train_df['transaction_date'].max()
test_min_date = test_df['transaction_date'].min()
print(f"Training set max date: {train_max_date}")
print(f"Test set min date: {test_min_date}")
print(f"Train/Test gap: {(test_min_date - train_max_date).days} days")
if test_min_date > train_max_date:
    print("✓ No temporal overlap - valid time-based split")
else:
    print("✗ WARNING: Temporal overlap detected!")

# Define features and target
target = 'items_sold'

# Identify numerical and categorical features (excluding date columns we engineered)
numerical_features = ['year', 'month', 'day_of_week', 'is_month_end', 'competition_density']
categorical_features = ['store_size', 'location_type', 'promotion_type']

# Note: store_id is excluded as it's an identifier
# is_weekend and is_festival are already binary numerical

print("\n" + "-"*40)
print("Features used for modeling:")
print("-"*40)
print(f"\nNumerical features: {numerical_features}")
print(f"Categorical features: {categorical_features}")
print(f"Target variable: {target}")

# Prepare X and y for train and test
X_train = train_df[numerical_features + categorical_features]
y_train = train_df[target]

X_test = test_df[numerical_features + categorical_features]
y_test = test_df[target]

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

## Task 3: Preprocessing Pipeline
print("="*60)
print("PREPROCESSING PIPELINE")
print("="*60)

# Create preprocessing pipelines for numerical and categorical features
numerical_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

print("\nPreprocessing pipeline components:")
print("-"*40)
print("\nNumerical Pipeline:")
print("  - StandardScaler: Centers to mean=0, scales to std=1")
print(f"  - Features: {numerical_features}")
print("\nCategorical Pipeline:")
print("  - OneHotEncoder (drop='first'): Creates dummy variables, avoids multicollinearity")
print(f"  - Features: {categorical_features}")
print("  - handle_unknown='ignore': Handles unseen categories in test set")

# Demonstrate the preprocessing pipeline
print("\n" + "-"*40)
print("Pipeline Demonstration (fitting on training data):")
print("-"*40)

# Fit the preprocessor on training data
preprocessor.fit(X_train)

# Transform both train and test sets
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\nOriginal X_train shape: {X_train.shape}")
print(f"Processed X_train shape: {X_train_processed.shape}")
print(f"\nOriginal X_test shape: {X_test.shape}")
print(f"Processed X_test shape: {X_test_processed.shape}")

# Get feature names after preprocessing
feature_names = (numerical_features + 
                 list(preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)))
print(f"\nTotal features after preprocessing: {len(feature_names)}")
print(f"Feature names: {feature_names[:10]}... (showing first 10)")

# Show sample of processed data
print("\n" + "-"*40)
print("Sample of processed training data (first 5 rows):")
print("-"*40)
processed_df = pd.DataFrame(X_train_processed[:5], columns=feature_names)
print(processed_df.round(3))

## Task 4: Model Training and Evaluation
print("="*60)
print("MODEL TRAINING AND EVALUATION")
print("="*60)

# Create pipelines with preprocessor and models
models = {
    'Linear Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
    ])
}

# Train and evaluate models
results = {}
predictions = {}

print("\nTraining models...\n")
for name, pipeline in models.items():
    print(f"Training {name}...")
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    predictions[name] = y_pred
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    }
    
    print(f"  ✓ {name} trained successfully")
    print(f"    RMSE: {rmse:.2f}")
    print(f"    MAE:  {mae:.2f}")
    print(f"    R²:   {r2:.4f}")
    print()

# Create results comparison table
print("="*60)
print("MODEL PERFORMANCE COMPARISON")
print("="*60)

results_df = pd.DataFrame(results).T
results_df = results_df.round(2)
print(results_df.to_string())

# Visualization: Parity Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[idx]
    
    # Scatter plot of actual vs predicted
    ax.scatter(y_test, y_pred, alpha=0.5, s=30, edgecolors='black', linewidth=0.5)
    
    # Diagonal reference line (perfect predictions)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    
    # Add metrics text on the plot
    rmse_val = results[name]['RMSE']
    mae_val = results[name]['MAE']
    ax.text(0.05, 0.95, f'RMSE: {rmse_val:.2f}\nMAE: {mae_val:.2f}', 
            transform=ax.transAxes, fontsize=11, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_xlabel('Actual Items Sold', fontsize=12)
    ax.set_ylabel('Predicted Items Sold', fontsize=12)
    ax.set_title(f'{name}\nParity Plot', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Feature Importance from Random Forest
print("="*60)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("="*60)

# Get the trained Random Forest model
rf_pipeline = models['Random Forest']
rf_model = rf_pipeline.named_steps['regressor']

# Get feature names after preprocessing
preprocessor_fitted = rf_pipeline.named_steps['preprocessor']
feature_names_processed = (numerical_features + 
                           list(preprocessor_fitted.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)))

# Get feature importances
feature_importance = pd.DataFrame({
    'feature': feature_names_processed,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importances (All Features):")
print(feature_importance.to_string(index=False))

# Top 5 most influential features
print("\n" + "-"*40)
print("TOP 5 MOST INFLUENTIAL FEATURES:")
print("-"*40)
top5 = feature_importance.head(5)
for idx, row in top5.iterrows():
    print(f"{idx+1}. {row['feature']:30s} - {row['importance']*100:.2f}% importance")

# Visualization: Feature Importance Bar Plot
plt.figure(figsize=(12, 8))

# Plot top 15 features for better visibility
top15 = feature_importance.head(15)
colors = plt.cm.viridis(np.linspace(0, 0.8, len(top15)))

plt.barh(range(len(top15)), top15['importance'].values, color=colors, edgecolor='black')
plt.yticks(range(len(top15)), top15['feature'].values)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title('Random Forest Feature Importance (Top 15 Features)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (idx, row) in enumerate(top15.iterrows()):
    plt.text(row['importance'] + 0.002, i, f'{row["importance"]*100:.1f}%', 
             va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Additional Analysis: Residual Analysis for Best Model
best_model_name = 'Random Forest' if results['Random Forest']['RMSE'] < results['Linear Regression']['RMSE'] else 'Linear Regression'
print(f"\nBest performing model: {best_model_name} (lower RMSE is better)")

# Residual plot for the best model
y_pred_best = predictions[best_model_name]
residuals = y_test - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
ax1 = axes[0]
ax1.scatter(y_pred_best, residuals, alpha=0.5, edgecolors='black', linewidth=0.5)
ax1.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax1.set_xlabel('Predicted Items Sold', fontsize=12)
ax1.set_ylabel('Residuals (Actual - Predicted)', fontsize=12)
ax1.set_title(f'{best_model_name} - Residual Plot', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Histogram of residuals
ax2 = axes[1]
ax2.hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
ax2.axvline(x=0, color='r', linestyle='--', linewidth=2)
ax2.set_xlabel('Residuals', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Residuals', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary Statistics
print("\n" + "="*60)
print("MODEL SUMMARY AND BUSINESS INSIGHTS")
print("="*60)

print(f"\n📊 DATA SUMMARY:")
print(f"   - Total transactions: {len(df)}")
print(f"   - Date range: {df['transaction_date'].min()} to {df['transaction_date'].max()}")
print(f"   - Average items sold per transaction: {df['items_sold'].mean():.1f}")

print(f"\n🎯 MODEL PERFORMANCE:")
for name, metrics in results.items():
    print(f"   - {name}: RMSE={metrics['RMSE']:.2f}, MAE={metrics['MAE']:.2f}, R²={metrics['R²']:.4f}")

print(f"\n🔑 KEY INSIGHTS FROM FEATURE IMPORTANCE:")
for idx, row in top5.iterrows():
    feature = row['feature']
    importance = row['importance'] * 100
    if 'promotion_type' in feature:
        promo_type = feature.split('_')[-1] if '_' in feature else feature
        print(f"   • {feature}: {importance:.1f}% - Promotion type significantly impacts sales")
    elif feature in ['is_festival', 'is_weekend']:
        print(f"   • {feature}: {importance:.1f}% - Timing/event factors matter")
    elif feature == 'competition_density':
        print(f"   • {feature}: {importance:.1f}% - Local competition affects demand")
    elif feature in ['month', 'day_of_week']:
        print(f"   • {feature}: {importance:.1f}% - Seasonal and weekly patterns exist")

print(f"\n💡 BUSINESS RECOMMENDATIONS:")
print("   1. Focus on high-impact promotion types identified by feature importance")
print("   2. Optimize inventory for peak days (high importance of day_of_week/month)")
print("   3. Adjust pricing/promotions based on competition density")
print("   4. Leverage festival periods (is_festival) for maximum sales impact")
print("   5. Use Random Forest for more accurate predictions (handles non-linear relationships)")

# Optional: Save predictions for further analysis
test_df_with_predictions = test_df.copy()
test_df_with_predictions['predicted_lr'] = predictions['Linear Regression']
test_df_with_predictions['predicted_rf'] = predictions['Random Forest']
test_df_with_predictions.to_csv('sales_predictions.csv', index=False)
print("\n✓ Predictions saved to 'sales_predictions.csv'")
